# Лабораторная работа: Рекомендательные системы

## Теоретическая часть

### 1. Суть задачи рекомендательных систем
Рекомендательные системы – это алгоритмы, которые анализируют поведение пользователей и предлагают персонализированные рекомендации товаров, фильмов, музыки и других объектов. Основная цель – предсказать предпочтения пользователей на основе имеющихся данных о взаимодействиях.


### 2. Метод коллаборативной фильтрации
Коллаборативная фильтрация (Collaborative Filtering, CF) – это метод рекомендаций, основанный на анализе поведения пользователей. Он работает на основе предположения, что пользователи с похожими предпочтениями в прошлом будут делать схожий выбор в будущем.

Существует два основных подхода:
1. **User-based CF** – рекомендации строятся на основе сходства пользователей.
2. **Item-based CF** – рекомендации строятся на основе сходства объектов.

### 3. Латентные факторные модели (Matrix Factorization)
Коллаборативная фильтрация может быть реализована через матричное разложение. Пусть у нас есть матрица взаимодействий пользователей и объектов R, где $( R_{u,i} )$ – оценка пользователя ( u ) для объекта ( i ). Тогда разложение можно представить в виде:
$$
R \approx U \cdot V^T
$$
где:
- ( U ) – матрица эмбеддингов пользователей,
- ( V ) – матрица эмбеддингов объектов.

Предсказание рейтинга рассчитывается как:
$$
\hat{R}_{u,i} = U_u \cdot V_i^T
$$

В данной лабораторной работе предполагается использование **нейросетевого метода**, который обучает эмбеддинги пользователей и объектов с помощью полносвязных слоев. Входные данные – индексы пользователей и объектов, которые преобразуются в векторные представления, а затем подаются на вход нейросети.


## Практическая часть
В данной работе вам предлагается реализовать рекомендательную систему на основе метода коллаборативной фильтрации, используя нейросетевую модель. Вы должны:
1. Подготовить данные: загрузить свой датасет (например, рейтинг фильмов, товаров, книг и т. д.).
2. Разбить данные на тренировочный и тестовый наборы.
3. Обучить модель, используя эмбеддинги пользователей и объектов.
4. Оценить качество модели на тестовом наборе.
5. Вывести список рекомендаций для выбранного пользователя.

In [1]:
# Импорты
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Определяем устройство (используем GPU, если доступно)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
# Определяем названия столбцов
columns = ['userID', 'songID', 'rating']
df = pd.read_csv('songsDataset.csv', sep=',', names=columns, skiprows=1)

# Ограничиваем датасет
df = df.sample(n=10000, random_state=42).reset_index(drop=True)

# Перекодировка ID в плотные индексы
user2idx = {u: i for i, u in enumerate(df['userID'].unique())}
song2idx = {s: i for i, s in enumerate(df['songID'].unique())}

df['user_idx'] = df['userID'].map(user2idx)
df['song_idx'] = df['songID'].map(song2idx)

In [3]:
# Определяем датасет PyTorch
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.songs = torch.tensor(df['song_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.songs[idx], self.ratings[idx]

In [4]:
# Определяем нейросетевую модель для коллаборативной фильтрации
class RecommenderNN(nn.Module):
    def __init__(self, num_users, num_songs, embedding_dim=32):
        super(RecommenderNN, self).__init__()
        # Эмбеддинги пользователей и фильмов
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.song_embedding = nn.Embedding(num_songs, embedding_dim)

        # Полносвязные слои для предсказания рейтинга
        self.fc_layers = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user, song):
        # Получаем эмбеддинги пользователя и фильма
        user_emb = self.user_embedding(user)
        song_emb = self.song_embedding(song)

        # Объединяем эмбеддинги
        x = torch.cat([user_emb, song_emb], dim=1)

        # Пропускаем через полносвязные слои
        return self.fc_layers(x).squeeze()

In [5]:
# Определяем количество пользователей и песней
num_users = df['user_idx'].nunique()
num_songs = df['song_idx'].nunique()

In [6]:
# Создаём датасеты и загрузчики данных
dataset = RatingsDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [7]:
# Инициализация модели
model = RecommenderNN(num_users, num_songs).to(device)

# Определяем функцию потерь (MSE) и оптимизатор (Adam)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)


In [8]:
for epoch in range(10):
    model.train()
    total_loss = 0
    all_predictions = []
    all_ratings = []
    for users, songs, ratings in train_loader:
        users, songs, ratings = users.to(device), songs.to(device), ratings.to(device)
        optimizer.zero_grad()
        predictions = model(users, songs)
        loss = criterion(predictions, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        all_predictions.extend(predictions.cpu().detach().numpy())
        all_ratings.extend(ratings.cpu().detach().numpy())

    # Средняя ошибка предсказания на тренировочной выборке
    rmse = math.sqrt(mean_squared_error(all_ratings, all_predictions))
    mae = mean_absolute_error(all_ratings, all_predictions)

    print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}')

# Оценка модели на тестовом наборе
model.eval()
test_predictions = []
test_ratings = []
with torch.no_grad():
    for users, songs, ratings in test_loader:
        users, songs, ratings = users.to(device), songs.to(device), ratings.to(device)
        predictions = model(users, songs)
        test_predictions.extend(predictions.cpu().numpy())
        test_ratings.extend(ratings.cpu().numpy())

# Средняя ошибка на тестовом наборе
test_rmse = math.sqrt(mean_squared_error(test_ratings, test_predictions))
test_mae = mean_absolute_error(test_ratings, test_predictions)

print(f'\nTest RMSE: {test_rmse:.4f}, Test MAE: {test_mae:.4f}')

# Обратное отображение индексов в реальные songID
idx2song = {i: s for s, i in song2idx.items()}

# Рекомендации для нескольких случайных пользователей
random_users = np.random.choice(df['userID'].unique(), size=5)

print("\nRecommendations for random users:")

for user_id in random_users:
    # получаем индекс пользователя
    user_idx = user2idx[user_id]

    user_tensor = torch.tensor([user_idx] * num_songs, dtype=torch.long).to(device)
    song_tensor = torch.arange(num_songs, dtype=torch.long).to(device)

    with torch.no_grad():
        predictions = model(user_tensor, song_tensor).cpu().numpy()

    top_songs_idx = predictions.argsort()[-5:][::-1]

    # переводим индексы обратно в реальные songID
    top_songs = [idx2song[i] for i in top_songs_idx]

    print(f"User {user_id}: Recommended songs {top_songs}")

Epoch 1, Loss: 2.9979582414627077
Epoch 2, Loss: 2.192800950050354
Epoch 3, Loss: 1.6783824558258056
Epoch 4, Loss: 1.129421534538269
Epoch 5, Loss: 0.6177982037067413
Epoch 6, Loss: 0.3028461799621582
Epoch 7, Loss: 0.15966116058826446
Epoch 8, Loss: 0.09487766024470329
Epoch 9, Loss: 0.0633557282090187
Epoch 10, Loss: 0.052705088824033734

Test RMSE: 1.8379, Test MAE: 1.5460

Recommendations for random users:
User 174504: Recommended songs [np.int64(19918), np.int64(67427), np.int64(128683), np.int64(7240), np.int64(2649)]
User 171323: Recommended songs [np.int64(18141), np.int64(73100), np.int64(118764), np.int64(28345), np.int64(27548)]
User 80057: Recommended songs [np.int64(51805), np.int64(9095), np.int64(68572), np.int64(89148), np.int64(13837)]
User 105448: Recommended songs [np.int64(84110), np.int64(83519), np.int64(78174), np.int64(34351), np.int64(62450)]
User 12718: Recommended songs [np.int64(83519), np.int64(8781), np.int64(62450), np.int64(2885), np.int64(96661)]
